In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import os

# ==========================================
# 1. DATA LOADING & ROBUST SCALING
# ==========================================
print("Loading and Scaling Data...")
train_raw = pd.read_csv('train.csv')
test_raw = pd.read_csv('test.csv')

# Use 'id' logic from before
if 'Id' not in test_raw.columns: test_raw['Id'] = test_raw.index

FEATURE_COLS = [f'f{i}' for i in range(26)]
TARGET_COL = 'y'
SYMBOL_COL = 'symbol_id'
SEQ_LEN = 10

# Calculate Robust Statistics on Training Data
medians = train_raw[FEATURE_COLS].median()
q1 = train_raw[FEATURE_COLS].quantile(0.25)
q3 = train_raw[FEATURE_COLS].quantile(0.75)
iqr = q3 - q1 + 1e-8

def robust_scale(df):
    df_out = df.copy()
    # 1. Median Fill
    df_out[FEATURE_COLS] = df_out[FEATURE_COLS].fillna(medians)
    # 2. IQR Scaling (Neural Nets need this to ignore outliers)
    df_out[FEATURE_COLS] = (df_out[FEATURE_COLS] - medians) / iqr
    # 3. Hard Clipping (Prevents Gradient Explosion)
    df_out[FEATURE_COLS] = df_out[FEATURE_COLS].clip(-5, 5)
    return df_out

train_df = robust_scale(train_raw).sort_values(['date_id', 'time_id'])
test_df = robust_scale(test_raw).sort_values([SYMBOL_COL, 'date_id', 'time_id'])

# ==========================================
# 2. DATASET
# ==========================================
class PFDataset(Dataset):
    def __init__(self, df, feature_cols, target_col, seq_len=10):
        self.data = []
        for _, group in df.groupby(SYMBOL_COL):
            if len(group) <= seq_len: continue
            X = group[feature_cols].values.astype(np.float32)
            y = group[target_col].values.astype(np.float32)
            for i in range(len(group) - seq_len):
                self.data.append((X[i:i+seq_len], y[i+seq_len]))
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        x, y = self.data[idx]
        return torch.from_numpy(x), torch.tensor(y)

# ==========================================
# 3. PFRWKV MODEL (REGULARIZED)
# ==========================================
class RWKV6_PF_Layer(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.time_decay = nn.Parameter(torch.ones(hidden_dim))
        self.time_first = nn.Parameter(torch.ones(hidden_dim))
        self.receptance = nn.Linear(input_dim, hidden_dim)
        self.key = nn.Linear(input_dim, hidden_dim)
        self.value = nn.Linear(input_dim, hidden_dim)
        self.ln = nn.LayerNorm(hidden_dim) # Crucial for stability

    def forward(self, x):
        B, T, D = x.size()
        r = torch.sigmoid(self.receptance(x))
        k = self.key(x)
        v = self.value(x)
        out = torch.zeros(B, T, self.hidden_dim).to(x.device)
        state = torch.zeros(B, self.hidden_dim).to(x.device)
        for t in range(T):
            state = state * torch.exp(-self.time_decay) + k[:, t, :] * v[:, t, :]
            out[:, t, :] = r[:, t, :] * (state + self.time_first * k[:, t, :])
        return self.ln(out)

class PFRWKV(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_particles=32):
        super().__init__()
        self.num_particles = num_particles
        self.rwkv = RWKV6_PF_Layer(input_dim, hidden_dim)
        
        # Particle System: Drift and Volatility
        self.particle_head = nn.Linear(hidden_dim, 2)
        
        self.fc_out = nn.Sequential(
            nn.Linear(hidden_dim + input_dim + 1, 64),
            nn.Dropout(0.2), # High regularization
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        B, T, D = x.size()
        rwkv_out = self.rwkv(x)
        last_hn = rwkv_out[:, -1, :]
        
        # Particle Filter logic
        params = self.particle_head(last_hn)
        drift = params[:, 0:1]
        vol = torch.exp(params[:, 1:2])
        
        # Stochastic resampling
        noise = torch.randn(B, self.num_particles).to(x.device)
        particles = drift + (noise * vol)
        z_filtered = particles.mean(dim=1, keepdim=True)
        
        last_x = x[:, -1, :]
        combined = torch.cat([last_x, last_hn, z_filtered], dim=1)
        pred = self.fc_out(combined).squeeze(-1)
        
        return pred, vol.mean()

# ==========================================
# 4. TRAINING
# ==========================================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = PFRWKV(len(FEATURE_COLS)).to(DEVICE)
loader = DataLoader(PFDataset(train_df, FEATURE_COLS, TARGET_COL), batch_size=4096, shuffle=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)

print("Training PFRWKV...")
for epoch in range(5):
    model.train()
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        pred, vol_loss = model(xb)
        # Total loss: Prediction error + Volatility penalty (keeps particles from exploding)
        loss = nn.MSELoss()(pred, yb) + 0.01 * vol_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    print(f"Epoch {epoch+1} complete.")

# ==========================================
# 5. SUBMISSION
# ==========================================
model.eval()
test_preds = []
print("Generating Submission...")
with torch.no_grad():
    for symbol, group in test_df.groupby(SYMBOL_COL):
        X_feat = torch.tensor(group[FEATURE_COLS].values, dtype=torch.float32).to(DEVICE)
        padded_X = torch.cat([X_feat[0:1].expand(SEQ_LEN-1, -1), X_feat], dim=0)
        windows = padded_X.unfold(0, SEQ_LEN, 1).transpose(1, 2)
        pred, _ = model(windows)
        test_preds.extend(pred.cpu().numpy().flatten())

submission = pd.DataFrame({'Id': test_df['Id'], 'y': test_preds})
submission.to_csv('submission.csv', index=False)
print("PFRWKV Submission Saved.")

NameError: name 'SYMBOL_COL' is not defined